# Retail Sales Data Analysis with AI

**Project:** Retail Sales Data Analysis with AI  
**Dataset:** Superstore-style retail sales dataset

In [ ]:
# 1. Install/import required libraries
# If needed in a fresh environment, uncomment the next line:
# %pip install pandas numpy matplotlib seaborn scikit-learn plotly

import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("Libraries imported successfully.")


In [ ]:
# 2. Load dataset
DATA_PATH = r"/mnt/data/bce89feb-5c46-442c-8afe-f420c0156c95.csv"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        "Dataset not found. Update DATA_PATH to the location of your CSV file."
    )

df = pd.read_csv(DATA_PATH)
df.columns = [c.strip() for c in df.columns]

print("Dataset shape:", df.shape)
display(df.head())


In [ ]:
# 3. Initial data inspection
print("Columns:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nMissing values:")
display(df.isna().sum().to_frame("missing_values"))

print("\nDuplicate rows:", df.duplicated().sum())


In [ ]:
# 4. Data cleaning and feature engineering
required_columns = ["Order Date", "Sales"]
missing_required = [c for c in required_columns if c not in df.columns]
if missing_required:
    raise ValueError(f"Required columns missing from dataset: {missing_required}")

df["Order Date"] = pd.to_datetime(df["Order Date"], dayfirst=True, errors="coerce")

if "Ship Date" in df.columns:
    df["Ship Date"] = pd.to_datetime(df["Ship Date"], dayfirst=True, errors="coerce")

df["Sales"] = pd.to_numeric(df["Sales"], errors="coerce")
df = df.dropna(subset=["Order Date", "Sales"]).copy()
df = df[df["Sales"] >= 0].copy()

df["Year"] = df["Order Date"].dt.year
df["Month"] = df["Order Date"].dt.month
df["MonthName"] = df["Order Date"].dt.strftime("%b")
df["Quarter"] = df["Order Date"].dt.quarter
df["DayOfWeek"] = df["Order Date"].dt.day_name()

if "Ship Date" in df.columns:
    df["Fulfillment Days"] = (df["Ship Date"] - df["Order Date"]).dt.days
else:
    df["Fulfillment Days"] = np.nan

print("Cleaned dataset shape:", df.shape)
display(df.head())


## 5. Key Business KPIs

The following KPIs provide a high-level view of retail performance:
- Total sales
- Number of orders
- Average order value
- Number of products
- Average fulfillment time (when shipping data is available)


In [ ]:
# 5. KPI summary
total_sales = df["Sales"].sum()

if "Order ID" in df.columns:
    total_orders = df["Order ID"].nunique()
    avg_order_value = df.groupby("Order ID")["Sales"].sum().mean()
else:
    total_orders = len(df)
    avg_order_value = df["Sales"].mean()

products_sold = df["Product ID"].nunique() if "Product ID" in df.columns else df["Product Name"].nunique() if "Product Name" in df.columns else np.nan
avg_fulfillment = df["Fulfillment Days"].mean()

kpis = pd.DataFrame({
    "Metric": ["Total Sales", "Total Orders", "Average Order Value", "Products Sold", "Average Fulfillment Days"],
    "Value": [total_sales, total_orders, avg_order_value, products_sold, avg_fulfillment]
})

display(kpis)


## 6. Sales Trend Analysis

In [ ]:
# Monthly sales trend
monthly_sales = (
    df.set_index("Order Date")
      .resample("ME")["Sales"]
      .sum()
      .reset_index()
)

plt.figure(figsize=(14, 5))
plt.plot(monthly_sales["Order Date"], monthly_sales["Sales"], marker="o")
plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Sales")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

display(monthly_sales.tail(12))


In [ ]:
# Year-over-year monthly comparison
month_order = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

monthly_year = (
    df.groupby(["Year", "Month", "MonthName"])["Sales"]
      .sum()
      .reset_index()
      .sort_values(["Year", "Month"])
)

plt.figure(figsize=(14, 6))
sns.lineplot(data=monthly_year, x="Month", y="Sales", hue="Year", marker="o")
plt.title("Monthly Sales by Year")
plt.xlabel("Month Number")
plt.ylabel("Sales")
plt.xticks(range(1, 13), month_order)
plt.tight_layout()
plt.show()


## 7. Seasonality Analysis

In [ ]:
# Sales by month across all years
seasonal_month = (
    df.groupby("Month")["Sales"]
      .sum()
      .reindex(range(1, 13), fill_value=0)
)

plt.figure(figsize=(12, 5))
plt.bar(month_order, seasonal_month.values)
plt.title("Total Sales by Calendar Month")
plt.xlabel("Month")
plt.ylabel("Sales")
plt.tight_layout()
plt.show()

# Sales by day of week
day_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
weekly_sales = df.groupby("DayOfWeek")["Sales"].sum().reindex(day_order)

plt.figure(figsize=(12, 5))
plt.bar(weekly_sales.index, weekly_sales.values)
plt.title("Sales by Day of Week")
plt.xlabel("Day")
plt.ylabel("Sales")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


## 8. Category and Sub-Category Analysis

In [ ]:
if "Category" in df.columns:
    category_sales = df.groupby("Category")["Sales"].sum().sort_values(ascending=False)
    display(category_sales.to_frame("Sales"))

    plt.figure(figsize=(9, 5))
    plt.bar(category_sales.index, category_sales.values)
    plt.title("Sales by Category")
    plt.xlabel("Category")
    plt.ylabel("Sales")
    plt.tight_layout()
    plt.show()

if "Sub-Category" in df.columns:
    subcategory_sales = df.groupby("Sub-Category")["Sales"].sum().sort_values(ascending=False)
    display(subcategory_sales.to_frame("Sales").head(15))

    plt.figure(figsize=(10, 6))
    top_sub = subcategory_sales.head(15).sort_values()
    plt.barh(top_sub.index, top_sub.values)
    plt.title("Top Sub-Categories by Sales")
    plt.xlabel("Sales")
    plt.tight_layout()
    plt.show()


## 9. Product Performance Analysis

In [ ]:
if "Product Name" in df.columns:
    product_sales = df.groupby("Product Name")["Sales"].sum().sort_values(ascending=False)

    print("Top 10 products:")
    display(product_sales.head(10).to_frame("Sales"))

    print("Bottom 10 products:")
    display(product_sales.tail(10).to_frame("Sales"))

    top10 = product_sales.head(10).sort_values()
    plt.figure(figsize=(10, 6))
    plt.barh(top10.index, top10.values)
    plt.title("Top 10 Products by Sales")
    plt.xlabel("Sales")
    plt.tight_layout()
    plt.show()


## 10. Regional, State, and Customer Segment Analysis

In [ ]:
# Region
if "Region" in df.columns:
    region_sales = df.groupby("Region")["Sales"].sum().sort_values(ascending=False)
    display(region_sales.to_frame("Sales"))

    plt.figure(figsize=(9, 5))
    plt.bar(region_sales.index, region_sales.values)
    plt.title("Sales by Region")
    plt.xlabel("Region")
    plt.ylabel("Sales")
    plt.tight_layout()
    plt.show()

# State
if "State" in df.columns:
    state_sales = df.groupby("State")["Sales"].sum().sort_values(ascending=False).head(10)
    plt.figure(figsize=(10, 6))
    plt.barh(state_sales.index[::-1], state_sales.values[::-1])
    plt.title("Top 10 States by Sales")
    plt.xlabel("Sales")
    plt.tight_layout()
    plt.show()

# Segment
if "Segment" in df.columns:
    segment_sales = df.groupby("Segment")["Sales"].sum().sort_values(ascending=False)
    display(segment_sales.to_frame("Sales"))

    plt.figure(figsize=(8, 5))
    plt.bar(segment_sales.index, segment_sales.values)
    plt.title("Sales by Customer Segment")
    plt.xlabel("Segment")
    plt.ylabel("Sales")
    plt.tight_layout()
    plt.show()


## 11. AI/ML Sales Forecasting

To add an AI/ML component, monthly sales are modeled as a supervised time-series regression problem.

### Features
- Year
- Month
- Quarter
- Time index
- Lag 1 month sales
- Lag 2 month sales
- Lag 3 month sales
- 3-month rolling average
- 6-month rolling average

A **Random Forest Regressor** is used because it can model non-linear relationships between the engineered time features and historical sales.

The final observations are reserved as a chronological test set to avoid randomly mixing future and past observations.


In [ ]:
# 11.1 Prepare monthly forecasting dataset
monthly = (
    df.set_index("Order Date")["Sales"]
      .resample("ME")
      .sum()
      .reset_index()
      .rename(columns={"Sales": "MonthlySales"})
)

monthly["Year"] = monthly["Order Date"].dt.year
monthly["Month"] = monthly["Order Date"].dt.month
monthly["Quarter"] = monthly["Order Date"].dt.quarter
monthly["TimeIndex"] = np.arange(len(monthly))

monthly["Lag1"] = monthly["MonthlySales"].shift(1)
monthly["Lag2"] = monthly["MonthlySales"].shift(2)
monthly["Lag3"] = monthly["MonthlySales"].shift(3)
monthly["RollingMean3"] = monthly["MonthlySales"].shift(1).rolling(3).mean()
monthly["RollingMean6"] = monthly["MonthlySales"].shift(1).rolling(6).mean()

model_data = monthly.dropna().copy()

feature_cols = [
    "Year", "Month", "Quarter", "TimeIndex",
    "Lag1", "Lag2", "Lag3", "RollingMean3", "RollingMean6"
]

X = model_data[feature_cols]
y = model_data["MonthlySales"]

print("Forecasting observations:", len(model_data))
display(model_data.tail())


In [ ]:
# 11.2 Chronological train/test split
test_size = max(3, int(round(len(model_data) * 0.20)))
train = model_data.iloc[:-test_size].copy()
test = model_data.iloc[-test_size:].copy()

X_train = train[feature_cols]
y_train = train["MonthlySales"]
X_test = test[feature_cols]
y_test = test["MonthlySales"]

model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    max_depth=10,
    min_samples_leaf=2
)

model.fit(X_train, y_train)
predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

metrics = pd.DataFrame({
    "Metric": ["MAE", "RMSE", "R²"],
    "Value": [mae, rmse, r2]
})

display(metrics)


In [ ]:
# 11.3 Actual vs predicted sales
comparison = test[["Order Date", "MonthlySales"]].copy()
comparison["PredictedSales"] = predictions
comparison["AbsoluteError"] = abs(comparison["MonthlySales"] - comparison["PredictedSales"])

display(comparison)

plt.figure(figsize=(14, 5))
plt.plot(comparison["Order Date"], comparison["MonthlySales"], marker="o", label="Actual")
plt.plot(comparison["Order Date"], comparison["PredictedSales"], marker="o", label="Predicted")
plt.title("Actual vs Predicted Monthly Sales")
plt.xlabel("Month")
plt.ylabel("Sales")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# 11.4 Feature importance
importance = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)

display(importance.to_frame("Importance"))

plt.figure(figsize=(10, 5))
plt.barh(importance.index[::-1], importance.values[::-1])
plt.title("Random Forest Feature Importance")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()


## 12. Future Sales Forecast

The model is recursively used to estimate the next six months. Each predicted month becomes an input lag for the following month.

**Note:** Forecasts are model-generated estimates based on historical sales patterns and should not be interpreted as guaranteed future revenue.


In [ ]:
# 12.1 Recursive six-month forecast
history = monthly[["Order Date", "MonthlySales"]].copy()
future_rows = []

for step in range(1, 7):
    next_date = history["Order Date"].max() + pd.offsets.MonthEnd(1)

    last_sales = history["MonthlySales"].tolist()
    lag1 = last_sales[-1]
    lag2 = last_sales[-2] if len(last_sales) >= 2 else lag1
    lag3 = last_sales[-3] if len(last_sales) >= 3 else lag2

    rolling3 = np.mean(last_sales[-3:]) if len(last_sales) >= 3 else np.mean(last_sales)
    rolling6 = np.mean(last_sales[-6:]) if len(last_sales) >= 6 else np.mean(last_sales)

    feature_row = pd.DataFrame([{
        "Year": next_date.year,
        "Month": next_date.month,
        "Quarter": next_date.quarter,
        "TimeIndex": len(history),
        "Lag1": lag1,
        "Lag2": lag2,
        "Lag3": lag3,
        "RollingMean3": rolling3,
        "RollingMean6": rolling6
    }])

    forecast_value = model.predict(feature_row[feature_cols])[0]

    new_row = pd.DataFrame([{
        "Order Date": next_date,
        "MonthlySales": forecast_value
    }])

    history = pd.concat([history, new_row], ignore_index=True)
    future_rows.append(new_row.iloc[0].to_dict())

future_forecast = pd.DataFrame(future_rows)

display(future_forecast)


In [ ]:
# 12.2 Plot historical and forecast sales
plt.figure(figsize=(14, 6))
plt.plot(monthly["Order Date"], monthly["MonthlySales"], marker="o", label="Historical Sales")
plt.plot(future_forecast["Order Date"], future_forecast["MonthlySales"], marker="o", linestyle="--", label="6-Month Forecast")
plt.title("Historical Sales and Six-Month AI Forecast")
plt.xlabel("Month")
plt.ylabel("Sales")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 13. Automated Business Insights

The following section converts the analysis into concise, data-driven observations. These are descriptive insights derived from the supplied dataset; they are not guarantees of future business outcomes.


In [ ]:
# 13.1 Generate business insights
insights = []

# Category insight
if "Category" in df.columns:
    cat = df.groupby("Category")["Sales"].sum().sort_values(ascending=False)
    insights.append(f"Highest-sales category: {cat.index[0]} with sales of {cat.iloc[0]:,.2f}.")
    insights.append(f"Lowest-sales category: {cat.index[-1]} with sales of {cat.iloc[-1]:,.2f}.")

# Region insight
if "Region" in df.columns:
    reg = df.groupby("Region")["Sales"].sum().sort_values(ascending=False)
    insights.append(f"Highest-sales region: {reg.index[0]} with sales of {reg.iloc[0]:,.2f}.")
    insights.append(f"Lowest-sales region: {reg.index[-1]} with sales of {reg.iloc[-1]:,.2f}.")

# Product insight
if "Product Name" in df.columns:
    prod = df.groupby("Product Name")["Sales"].sum().sort_values(ascending=False)
    insights.append(f"Top product by sales: {prod.index[0]} with sales of {prod.iloc[0]:,.2f}.")
    insights.append(f"Lowest-sales product in the dataset: {prod.index[-1]} with sales of {prod.iloc[-1]:,.2f}.")

# Seasonality
season = df.groupby("Month")["Sales"].sum()
peak_month = int(season.idxmax())
low_month = int(season.idxmin())
insights.append(f"Highest-sales calendar month across the available years: {month_order[peak_month-1]}.")
insights.append(f"Lowest-sales calendar month across the available years: {month_order[low_month-1]}.")

# Forecast
if len(future_forecast):
    total_forecast = future_forecast["MonthlySales"].sum()
    insights.append(f"Model-estimated sales for the next six months: {total_forecast:,.2f}.")

for i, insight in enumerate(insights, 1):
    print(f"{i}. {insight}")


## 14. Final Project Summary

### Key outcomes
- Historical retail sales were cleaned and transformed into analysis-ready data.
- Sales trends and seasonality were explored at monthly and weekly levels.
- Product, category, sub-category, regional, state, and customer-segment performance were analyzed.
- A Random Forest machine-learning model was trained for monthly sales forecasting.
- Model performance was evaluated using MAE, RMSE, and R².
- A six-month recursive sales forecast was generated.
- Automated business insights were produced from the dataset.

### Technologies used
- Python
- Pandas
- NumPy
- Matplotlib
- Seaborn
- Plotly
- Scikit-learn
- Jupyter Notebook
